In [16]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
%matplotlib inline
%matplotlib qt
import matplotlib.pyplot as plt
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [ ]:
import torchvision
import torchvision.transforms as transforms
import torch

train_set = torchvision.datasets.FashionMNIST(
    root="./data/FashionMNIST",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

test_set = torchvision.datasets.FashionMNIST(
    root="./data/FashionMNIST",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

index = 0

image, label = train_set[index]

# -----------------------------------
# Print info
# -----------------------------------

print("Training Samples:", len(train_set))
print("Test Samples:", len(test_set))
print("Index:", index)
print("Label:", label)
print("Tensor shape:", image.shape)

In [ ]:
batch_size = 100

train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False
)

In [24]:
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

class Network(nn.Module):
    def __init__(self):
        super(Network, self).__init__()
        
        #Layer takes one input, has a kernel/filter of size 5 and outputs 5 featuremaps
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5)
        #Next layer taking in input of val 1
        self.conv2 = nn.Conv2d(in_channels=6, out_channels=12, kernel_size=5)
        
        self.fc1 = nn.Linear(in_features=12*4*4, out_features=120)
        self.fc2 = nn.Linear(in_features=120, out_features=60)
        self.out = nn.Linear(in_features=60, out_features=10)

    def forward(self, t):
        #Implement forward pass
        t=t
        
        t=self.conv1(t)
        t=F.relu(t)
        t=F.max_pool2d(t, kernel_size=2, stride=2)
        
        t=self.conv2(t)
        t=F.relu(t)
        t=F.max_pool2d(t, kernel_size=2, stride=2)
        
        
        t=t.reshape(-1,12 * 4 * 4)
        t=self.fc1(t)
        t=F.relu(t)
        
        t=self.fc2(t)
        t=F.relu(t)
        
        t=self.out(t)
       # t=F.softmax(t, dim=1)
        
        return t

In [25]:
from itertools import product

parameters = dict(
    lr=[.01,.001],
    batch_size=[10,100,1000],
    shuffle=[True,False]
)

paramValues = [
    v for v in parameters.values()
]

for lr, batch_size, shuffle in product(*paramValues):
    print(
        lr,
        batch_size,
        shuffle
    )

0.01 10 True
0.01 10 False
0.01 100 True
0.01 100 False
0.01 1000 True
0.01 1000 False
0.001 10 True
0.001 10 False
0.001 100 True
0.001 100 False
0.001 1000 True
0.001 1000 False


In [26]:
network = Network().to(device)
optimizer = optim.Adam(network.parameters(), lr=0.01)
def get_num_correct(prediction, label):
    return prediction.argmax(dim=1).eq(label).sum().item()

In [27]:
from torch.utils.tensorboard import SummaryWriter
images, labels = next(iter(train_loader))

grid = torchvision.utils.make_grid(images)

comment = f"batch_size={batch_size}_lr={lr}"

tb = SummaryWriter(comment=comment)

tb.add_image(
    "FashionMNIST Images",
    grid
)

tb.add_graph(
    network,
    images.to(device)
)

In [28]:
#Training loop
batch_size = 100
lr = 0.01
network = Network().to(device)

training_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False
)

optimizer = optim.Adam(network.parameters(), lr=lr)

images, labels = next(iter(training_loader))
grid = torchvision.utils.make_grid(images)

comment = f'batch_size={batch_size}, lr = {lr}'
tb= SummaryWriter(comment=comment)

tb.add_image('images', grid)
tb.add_graph(network, images.to(device))

for epoch in range(5):

    network.train()

    totalLoss = 0
    totalCorrect = 0
    
    for batch in training_loader:
        images, labels = batch

        images = images.to(device)
        labels = labels.to(device)

        pred = network(images)
        loss = F.cross_entropy(pred, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        totalLoss += loss.item() * batch_size
        totalCorrect += get_num_correct(pred, labels)

    network.eval()

    testCorrect = 0

    with torch.no_grad():

        for batch in test_loader:
            images, labels = batch

            images = images.to(device)
            labels = labels.to(device)

            pred = network(images)

            testCorrect += get_num_correct(
                pred,
                labels
            )

    testAccuracy = testCorrect / len(test_set)
    
    tb.add_scalar('Loss:', totalLoss, epoch)
    tb.add_scalar('Correct:', totalCorrect, epoch)
    tb.add_scalar('Accuracy:', totalCorrect/len(train_set), epoch)
    tb.add_scalar('Accuracy/Test:', testAccuracy, epoch)
    
    tb.add_histogram('conv1.bias', network.conv1.bias.cpu(), epoch)
    tb.add_histogram('conv1.weight', network.conv1.weight.cpu(), epoch)
    tb.add_histogram('conv1.weight.grad', network.conv1.weight.grad.cpu(), epoch)

    print(
        "epoch:",
        epoch,
        "Total Correct:",
        totalCorrect,
        "Train Accuracy:",
        totalCorrect/len(train_set),
        "Test Accuracy:",
        testAccuracy,
        "Loss:",
        totalLoss
    )

tb.close()

epoch: 0 Total Correct: 47833 Train Accuracy: 0.7972166666666667 Test Accuracy: 0.8427 Loss: 32431.260696053505
epoch: 1 Total Correct: 51581 Train Accuracy: 0.8596833333333334 Test Accuracy: 0.8589 Loss: 22797.4613904953
epoch: 2 Total Correct: 52370 Train Accuracy: 0.8728333333333333 Test Accuracy: 0.8558 Loss: 20446.499317884445
epoch: 3 Total Correct: 52763 Train Accuracy: 0.8793833333333333 Test Accuracy: 0.872 Loss: 19645.35419344902
epoch: 4 Total Correct: 53034 Train Accuracy: 0.8839 Test Accuracy: 0.8657 Loss: 19000.502583384514


In [29]:
torch.save(
    network.state_dict(),
    "fashion_cnn.pth"
)

print("Model Saved")

Model Saved


In [30]:
all_preds = torch.empty((0, 10))

network.eval()

with torch.no_grad():

    for batch in test_loader:

        images, labels = batch

        images = images.to(device)

        preds = network(images)

        all_preds = torch.cat(
            (all_preds, preds.cpu()),
            dim=0
        )

In [31]:
preds_correct = get_num_correct(
    all_preds,
    test_set.targets
)

print(
    'Correct:',
    preds_correct
)

print(
    'Accuracy:',
    preds_correct / len(test_set)
)

Correct: 8657
Accuracy: 0.8657


In [32]:
stacked = torch.stack(
    (
        test_set.targets,
        all_preds.argmax(dim=1)
    ),
    dim=1
)

cmt = torch.zeros(
    10,
    10,
    dtype=torch.int32
)

for p in stacked:
    j, k = p.tolist()
    cmt[j, k] = cmt[j, k] + 1

cmt

tensor([[728,   3,  14,  32,   9,   1, 207,   0,   6,   0],
        [  1, 970,   3,  19,   2,   0,   4,   0,   1,   0],
        [ 11,   0, 783,  11, 106,   0,  85,   0,   4,   0],
        [ 16,   7,   7, 902,  25,   0,  41,   0,   1,   1],
        [  0,   2,  83,  38, 802,   0,  75,   0,   0,   0],
        [  1,   0,   0,   0,   0, 916,   0,  26,   1,  56],
        [ 82,   0,  84,  37,  91,   0, 694,   0,  12,   0],
        [  0,   0,   0,   0,   0,   7,   0, 929,   0,  64],
        [  2,   1,   6,   1,   5,   3,  21,   6, 954,   1],
        [  0,   0,   0,   0,   0,   2,   1,  18,   0, 979]], dtype=torch.int32)

In [33]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))

sns.heatmap(
    cmt,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('FashionMNIST Confusion Matrix')

plt.show()

In [34]:
network.eval()

with torch.no_grad():

    images, labels = next(iter(test_loader))

    images = images.to(device)
    labels = labels.to(device)

    preds = network(images)

    preds = preds.argmax(dim=1)

incorrect = preds.ne(labels)

print(
    "Incorrect Predictions:",
    incorrect.sum().item()
)

Incorrect Predictions: 15


In [35]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Laptop GPU
